# 02. 테일러 전개와 선형화

비선형 로봇 모델을 한 점 근처에서 선형 모델로 근사하는 것이 선형화다.
EKF, LQR, 비선형 최적화는 모두 이 아이디어 위에 있다.

$$f(x) \approx f(x_0) + J_f(x_0)(x-x_0)$$

2차까지 쓰면 Hessian이 등장한다:

$$f(x) \approx f(x_0) + \nabla f(x_0)^T(x-x_0) + \frac{1}{2}(x-x_0)^T H(x_0)(x-x_0)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 1차/2차 테일러 근사

$f(x)=\sin(x)$ 를 $x_0=0.8$ 근처에서 근사한다.

In [ ]:
def f(x): return np.sin(x)
def df(x): return np.cos(x)
def d2f(x): return -np.sin(x)

x0 = 0.8
x = np.linspace(-1.0, 2.6, 300)
first = f(x0) + df(x0) * (x - x0)
second = first + 0.5 * d2f(x0) * (x - x0)**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x, f(x), color='#534AB7', lw=2.5, label='sin(x)')
axes[0].plot(x, first, '--', color='#E85D24', lw=2, label='1차 근사')
axes[0].plot(x, second, '--', color='#1D9E75', lw=2, label='2차 근사')
axes[0].scatter([x0], [f(x0)], s=70, color='black', zorder=5)
axes[0].set_ylim(-1.2, 1.4)
axes[0].grid(alpha=0.25); axes[0].legend()
axes[0].set_title('Taylor approximation')

axes[1].plot(x, np.abs(f(x) - first), color='#E85D24', lw=2, label='1차 오차')
axes[1].plot(x, np.abs(f(x) - second), color='#1D9E75', lw=2, label='2차 오차')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.25); axes[1].legend()
axes[1].set_title('근사점에서 멀어질수록 오차 증가')
plt.tight_layout()
plt.savefig('assets/02_taylor_approximation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Differential Drive 로봇 모델 선형화

상태 $x=[p_x,p_y,\theta]^T$, 입력 $u=[v,\omega]^T$:

$$\dot{x}=\begin{bmatrix}v\cos\theta\\v\sin\theta\\\omega\end{bmatrix}$$

연속 시간 Jacobian:

$$A=\frac{\partial f}{\partial x}, \qquad B=\frac{\partial f}{\partial u}$$

In [ ]:
def f_robot(x, u):
    px, py, th = x
    v, w = u
    return np.array([v*np.cos(th), v*np.sin(th), w])

def linearize_robot(x, u):
    _, _, th = x
    v, _ = u
    A = np.array([
        [0.0, 0.0, -v*np.sin(th)],
        [0.0, 0.0,  v*np.cos(th)],
        [0.0, 0.0,  0.0]
    ])
    B = np.array([
        [np.cos(th), 0.0],
        [np.sin(th), 0.0],
        [0.0,        1.0]
    ])
    return A, B

x0 = np.array([1.0, 0.5, np.deg2rad(35)])
u0 = np.array([1.2, 0.4])
A, B = linearize_robot(x0, u0)

print('A = df/dx')
print(np.round(A, 4))
print('\nB = df/du')
print(np.round(B, 4))

# perturbation test
rng = np.random.default_rng(4)
dx = np.array([0.03, -0.02, np.deg2rad(2.0)])
du = np.array([0.05, -0.03])
true_delta = f_robot(x0 + dx, u0 + du) - f_robot(x0, u0)
lin_delta = A @ dx + B @ du

print('\n실제 변화:', np.round(true_delta, 6))
print('선형 근사:', np.round(lin_delta, 6))
print('오차:', np.round(true_delta - lin_delta, 6))

## 3. 선형화 오차 지도

선형화는 근사점 근처에서만 좋다. 특히 heading 오차가 커지면 삼각함수 비선형성이 커진다.

In [ ]:
th_offsets = np.deg2rad(np.linspace(-60, 60, 121))
v_offsets = np.linspace(-0.6, 0.6, 121)
Err = np.zeros((len(v_offsets), len(th_offsets)))

for i, dv in enumerate(v_offsets):
    for j, dth in enumerate(th_offsets):
        dx = np.array([0.0, 0.0, dth])
        du = np.array([dv, 0.0])
        true_delta = f_robot(x0 + dx, u0 + du) - f_robot(x0, u0)
        lin_delta = A @ dx + B @ du
        Err[i, j] = np.linalg.norm(true_delta - lin_delta)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.contourf(np.rad2deg(th_offsets), v_offsets, Err, levels=30, cmap='viridis')
plt.colorbar(im, ax=ax, label='linearization error')
ax.set_xlabel('heading perturbation Δθ (deg)')
ax.set_ylabel('velocity perturbation Δv')
ax.set_title('근사점에서 멀어질수록 선형화 오차 증가')
plt.savefig('assets/02_linearization_error.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| 1차 테일러 | $f(x)\approx f(x_0)+J\Delta x$ | EKF, LQR, 비선형 모델 근사 |
| Jacobian | $\partial f/\partial x$ | 상태/측정 모델 민감도 |
| Hessian | $\partial^2 f/\partial x^2$ | Newton method, 곡률 기반 최적화 |
| 선형화 오차 | 근사점에서 멀수록 증가 | 재선형화, 작은 시간 간격 필요 |